# Batch L1C → L2A (GRS 2.2) → L2B (GRSl2bgen) from the CNES datalake on TREX

This notebook processes the Sentinel-2 L1C images of the **CNES datalake** (S3 object storage of the HPC
cluster TREX) with **GRS 2.2** (L2A) and the latest **GRSl2bgen** (L2B), image by image. It follows the TREX
guide of the L1C-L2B chain (Overleaf *L1C-L2B processing chain on TREX*): same folders on
`/work/datalake/watcal`, auxiliary data (daily CAMS, COP-DEM GLO-30), environment rules and SLURM
settings. The products are written to the tree of the guide, the GRS version in their names:

```
/work/datalake/watcal/Sentinel-2/
    L1C/<tile>/YYYY/MM/DD/<name>.SAFE            images synchronised by sync_l1c.py (used as is if found)
    L2A/<tile>/YYYY/MM/DD/<S2x_MSIL2AGRS_..._V2.2.x>/<name>.nc, <name>_anc.nc, log_file.log
    L2B/<tile>/YYYY/MM/DD/<S2x_MSIL2B_...>/<name>.nc, <name>.json (STAC Item), <name>.log
    batch/                                       records of the runs, out of the product folders:
        profiles/<tile>/<image>_profile.json     time and memory of each step of an image
        summaries/<tile>/l1c_l2b_<tile>_<start>-<end>_<time>[_job<id>_<task>].csv
                                                 summary of each run (one file per tile)
```

{doc}`grs_datalake_trex_batch_v3` is the same batch with GRS 3.x.

1. **Configuration**: tiles, dates, datalake access, output folders, auxiliary data, options of the two
   processors. The tiles and dates can be overridden by environment variables, so the same notebook runs
   as a SLURM job (section 8).
2. **Datalake**: connection check and a look at the layout of the L1C bucket.
3. **List the L1C images** of the tiles in the date range, from the product names alone (any folder
   layout, `.SAFE` folders or `.zip` archives). Duplicates of a datatake (reprocessed with a newer
   baseline) are dropped. The cloud cover is then read for the selected images only, in parallel.
4. **Plan**: CAMS file of the day, DEM, names of the L2A and L2B products, and which products already
   exist (checked from file metadata only, to spare the Lustre metadata servers). The L1C products
   archived on tier 3 are checked, and their restoration is requested.
5. **Run**: each image is copied from the bucket to a local work folder (unzipped if needed) and
   processed. The local copy is then removed. Finished products are skipped, so the batch can be stopped
   and restarted at any time. A time budget makes the batch stop cleanly before the end of a SLURM job.
   Images without clear water are marked and not tried again.
6. **Summary** and **profiles**.
7. **Quick look** at the products of one image.
8. **SLURM job array** on a tile × month grid, as `run_l2a_array_2d.slurm` of the guide, running this
   notebook without Jupyter.

The functions come from {mod}`grstbx.batch` ({class}`~grstbx.batch.Batch`).

**Environments.** The notebook runs in the GRS 2.2 conda environment, **`GRSv2`** by default (`conda_env`, Jupyter kernel of that
environment): GRS 2.2, GRSl2bgen, grstbx and `s3fs`, which lists and downloads the images. The steps
specific to the CNES datalake run in the conda env **`s3-env`** of the computing centre
(`/softs/rh8/conda-envs/awss3-env`, ready with `assumerole`, `amaltheerestore` and `boto3`), called by the
notebook as a separate Python process: the temporary credentials of the role, and the restoration of the
images archived on tier 3. Nothing has to be installed for the datalake, except `s3fs` in `GRSv2`
(`pip install s3fs`).

```{note}
The access to the datalake follows the HPC wiki
([Consommateurs de données](https://hpc.pages.cnes.fr/wiki-hpc-sphinx/datalake-consommateur.html)): temporary
credentials of a read-only role, renewed during the batch. Set `role_arn`, then check the bucket and
the layout of the L1C products: section 2 lists the first levels of the bucket to help you set them.
```

```{important}
TREX loads its software with Spack modules, which put system paths in `PYTHONPATH`: start Jupyter (and every
job) with `unset PYTHONPATH; export PYTHONNOUSERSITE=1`, and with `MALLOC_ARENA_MAX=2` so that memory is reused
from one image to the next. The first cell checks it.
```

## 0. Settings

What to set before a run; everything else has working defaults (section 1). The tiles, dates, output root and
environment can be overridden by environment variables (`GRS_TILES`, `GRS_START`, `GRS_END`, `GRS_OUT_ROOT`,
`GRS_CONDA_ENV`): the tasks of the SLURM array (section 8) use them to give each task its tile and month.

| Setting | Description |
|---|---|
| `tiles`, `start_date`, `end_date` | Sentinel-2 tiles and date range (inclusive) |
| `out_root` | root of the L1C, L2A and L2B products on `/work/datalake` |
| `role_arn` | role of the CNES datalake to take on to read the L1C images (`/softs/tools/datalake/roles_list.sh`); default: the public read-only role of the OT account (env. `DATALAKE_ROLE_ARN`) |
| `OT_ACCOUNT` | id of the OT account (in the ARNs listed by `roles_list.sh`): default role, and restorations from tier 3 with `amaltheerestore` (env. `DATALAKE_OT_ACCOUNT`) |
| `account`, `hours` | SLURM account of your project and time limit of each task (tile x month) |
| `workers` | images processed at the same time, each in its own process (section 5; env. `GRS_WORKERS`) |
| `mem_per_image` | memory of one image (GB), for the memory asked by the SLURM tasks: `workers × mem_per_image`, 64 GB at least (section 8) |
| `conda_env` | conda env of the processing (GRS 2.2, GRSl2bgen, grstbx, `s3fs`), activated by the SLURM tasks |


In [ ]:
import os


def env_list(name, default):
    value = os.environ.get(name)
    return [v.strip() for v in value.split(',') if v.strip()] if value else default


# what to process
tiles = env_list('GRS_TILES', ['31TCJ', '24MWV', '24MWU', '20NHQ'])
start_date = os.environ.get('GRS_START', '2025-06-01')
end_date = os.environ.get('GRS_END', '2025-06-30')
out_root = os.environ.get('GRS_OUT_ROOT', '/work/datalake/watcal/Sentinel-2')

# CNES datalake: id of the OT account (in the ARNs of /softs/tools/datalake/roles_list.sh), and role taken on
# to read the L1C images (default: the public read-only role of the OT account)
OT_ACCOUNT = os.environ.get('DATALAKE_OT_ACCOUNT', '<OT account id>')
role_arn = os.environ.get('DATALAKE_ROLE_ARN', f'arn:aws:iam::{OT_ACCOUNT}:role/public-read-only-OT')

# SLURM job array (section 8)
account = 'obs2co'         # SLURM account of your project
hours = 4                  # time limit of a task (tile x month)

# images processed at the same time (each in its own process): memory and cores of a task scale with it
workers = int(os.environ.get('GRS_WORKERS', 1))
mem_per_image = 32         # GB per image at 20 m (GRS 2.2: about 30; see the profiles, section 6)
conda_env = os.environ.get('GRS_CONDA_ENV', 'GRSv2')   # conda env of the processing (GRS 2.2, GRSl2bgen, grstbx, s3fs)


In [ ]:
import os
import re
import sys
import time
import json
import shutil
import zipfile
import subprocess
import logging
import warnings
from concurrent.futures import ThreadPoolExecutor
from importlib.resources import files

import numpy as np
import pandas as pd
import xarray as xr
import yaml
import fsspec
import matplotlib.pyplot as plt
try:
    import s3fs  # noqa: F401, S3 file system of fsspec
except ImportError:
    raise ImportError('s3fs is needed in this environment to read the datalake: pip install s3fs')

import grs
import GRSl2bgen
from grstbx import batch

opj = os.path.join

logging.getLogger().setLevel(logging.INFO)    # logging.WARNING for a quieter batch
logging.getLogger('botocore').setLevel(logging.WARNING)
logging.getLogger('s3fs').setLevel(logging.WARNING)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='The specified chunks separate the stored chunks')

print(f'python {sys.version.split()[0]}, grs: {grs.__version__}, GRSl2bgen: {GRSl2bgen.__version__}')
if not grs.__version__.startswith('2.2'):
    warnings.warn(f'this notebook is set up for GRS 2.2 (NetCDF L2A products), not {grs.__version__}: '
                  'use grs_datalake_trex_batch_v3.ipynb for GRS 3.x')
# environment rules of TREX (guide, section 2.1)
if os.environ.get('PYTHONPATH'):
    warnings.warn(f"PYTHONPATH is set ({os.environ['PYTHONPATH']}): Spack modules may shadow the packages of this "
                  'environment; start Jupyter after `unset PYTHONPATH; export PYTHONNOUSERSITE=1`')
if os.environ.get('MALLOC_ARENA_MAX') is None:
    print('hint: start Jupyter with MALLOC_ARENA_MAX=2 so that memory is reused from one image to the next')

## 1. Configuration

The options of the processing (the settings of a run are in section 0).

| Parameter | Description |
|---|---|
| `out_root`, `l2a_dir`, `l2b_dir` | roots of the L2A and L2B products on `/work/datalake` (env. `GRS_OUT_ROOT`) |
| `batch_dir` | records of the runs, out of the product folders: profiles of the images and summaries of the runs, by tile (section 6) |
| `work_dir` | local folder for the copy of the image being processed (1 to 2 GB per image); default `$TMPDIR` of the SLURM job, else in your scratch space |
| `cams_file_templates` | CAMS files of the day, tried in turn (formatted with the acquisition `date`): daily archive of the datalake, then its fallback (mandatory with GRS 2.2) |
| `dem_file_template` | COP-DEM GLO-30 GeoTIFF of the tile, formatted with `tile`; `None` or missing file: no DEM |
| `resolution`, `max_cloud_cover`, `opac_model`, `scale_aot`, `levname`, `suffix` | GRS options, as in {doc}`../../tutorials/batch/grs_s2_batch_v2` |
| `persist_input` | GRSl2bgen: keep the masked L2A raster in memory (faster, more RAM) |
| `quick_check` | products checked from file metadata only (size, STAC Item), not opened: much lighter for the Lustre metadata servers on large batches (guide, troubleshooting matrix) |
| `overwrite_l2a`, `overwrite_l2b` | recompute existing products |
| `max_hours` | time budget of the run: no new image is started past it (env. `GRS_MAX_HOURS`; `None`: no limit) |

The example tiles cover four kinds of water:

| Tile | Area |
|---|---|
| `31TCJ` | Garonne, around Toulouse (France): rivers and reservoirs |
| `24MWV`, `24MWU` | coast of Ceará (Brazil): turbid coastal waters and reservoirs |
| `20NHQ` | Lesser Antilles, around Guadeloupe: clear tropical waters |

In [ ]:

# outputs, on the shared file system (tree of the TREX guide)
l2a_dir = opj(out_root, 'L2A')
l2b_dir = opj(out_root, 'L2B')
batch_dir = opj(out_root, 'batch')    # profiles and summaries of the runs, by tile

# local copy of the image being processed: node-local $TMPDIR in a SLURM job
user = os.environ.get('USER', 'user')
work_dir = opj(os.environ.get('TMPDIR') or f'/work/scratch/data/{user}', 'grs_batch_work')

# auxiliary data on TREX (guide, section 4.2): daily CAMS global forecasts, then the fallback archive
CAMS_DAILY = '{date:%Y/%m/%d}/{date:%Y-%m-%d}-cams-global-atmospheric-composition-forecasts.nc'
cams_file_templates = ['/work/datalake/static_aux/CAMS/OBS2CO_GRS_v2/' + CAMS_DAILY,
                       '/work/datalake/static_aux/CAMS/fallback_v2/' + CAMS_DAILY]
dem_file_template = '/work/datalake/static_aux/MNT/COP-DEM_GLO-30-DGED_S2_tiles/COP-DEM_GLO-30-DGED_{tile}.tif'

# GRS 2.2 (L2A, NetCDF)
resolution = 20         # m
max_cloud_cover = 0.8   # 0-1, Level-1 cloud cover
opac_model = None       # e.g. 'DESE_rh70' to force desert dust
scale_aot = 1
levname = 'L2AGRS'
suffix = None           # None: '_V<grs version>'

# GRSl2bgen (L2B, NetCDF)
persist_input = False


quick_check = True      # products checked from file metadata only (Lustre)
overwrite_l2a = False
overwrite_l2b = False
max_hours = float(os.environ['GRS_MAX_HOURS']) if os.environ.get('GRS_MAX_HOURS') else None

print(f'tiles {tiles}, {start_date} to {end_date}')
print(f'L2A -> {l2a_dir}\nL2B -> {l2b_dir}\nwork_dir: {work_dir}\nconda env of the SLURM tasks: {conda_env}')

### Datalake

The L1C images are read from the S3 object storage of the datalake (endpoint
`https://s3.datalake.cnes.fr`, region `us-east-1`) through `s3fs`. As described in
[Consommateurs de données](https://hpc.pages.cnes.fr/wiki-hpc-sphinx/datalake-consommateur.html), there is no
permanent key: your IPA account **takes on a read-only role** and gets **temporary credentials** (access key,
secret key and session token), valid 1 h by default and 12 h at most.

- `role_arn`: ARN of the role to take on (env. `DATALAKE_ROLE_ARN`), e.g. the public read-only role of the **OT**
  account (Earth observation data), as in `sync_l1c.py` of the guide, with the id of that account in
  `OT_ACCOUNT` (env. `DATALAKE_OT_ACCOUNT`);
  `/softs/tools/datalake/roles_list.sh` lists the roles your IPA account can take on. The credentials are
  fetched for `role_hours` hours (1 to 12) and **renewed `refresh_margin` minutes before they expire**, so a
  batch can run longer than the validity of one set of credentials.
- `role_arn = None`: credentials from the environment, e.g. written to `~/.aws/credentials` by
  `/softs/tools/datalake/use_role.sh -a <ARN> -t 43200` and selected by `AWS_PROFILE`. They are not renewed,
  so the batch must end within their validity (`max_hours` below 12).

The credentials are fetched by `assumerole.getCredentials`, run in `s3-env` (section 1b) from your Kerberos
ticket. **Setup** (once, on a TREX login node): a Kerberos ticket in a file, readable from the compute nodes
too (valid about a week: run `kinit` again then):

```bash
export KRB5CCNAME=FILE:/home/tmp/krb5cc_$(id -u)
kinit && klist
```

The Sentinel-2 L1C images of the OT account are in the bucket `sentinel2-l1c`, one `.zip` archive per
product, in a tree by tile, split into UTM zone, latitude band and square, then by date:

```
sentinel2-l1c/28/P/FV/2020/01/02/S2B_MSIL1C_20200102T111349_N0500_R137_T28PFV_20230425T035538.zip
```

`l1c_layout` is the glob pattern of the products, formatted with `root`, `tile`, `zone`, `band`, `square`
(`28`, `P`, `FV` for `28PFV`) and `year`: the images are then selected from their names, so a pattern matching
too much is harmless (only slower to list). Products can be `.zip` / `.SAFE.zip` archives or `.SAFE` folders.

`l1c_cache` is the folder of the L1C images already synchronised by stage 0 of the guide (`sync_l1c.py`,
`<tile>/YYYY/MM/DD/<name>.SAFE`): an image found there is processed from it, without download nor restoration.

Old products can be stored on **tier 3** (archive storage, `StorageClass` `glacier-*`): they must be
**restored** before they can be downloaded (section 4b). `restore_days` is how long a restored copy stays
available (10 days at most).

In [ ]:
endpoint_url = 'https://s3.datalake.cnes.fr'
role_hours = 4          # validity of the temporary credentials (h, 12 at most)
refresh_margin = 30     # renewed this many minutes before they expire

l1c_root = 's3://sentinel2-l1c'      # bucket of the Sentinel-2 L1C images (OT account)
# <zone>/<band>/<square>/YYYY/MM/DD/<product>.zip, e.g. sentinel2-l1c/28/P/FV/2020/01/02/S2B_MSIL1C_..._T28PFV_....zip
l1c_layout = '{root}/{zone}/{band}/{square}/{year}/*/*/S2*_MSIL1C_*.zip'
# images synchronised by sync_l1c.py (stage 0 of the guide), <tile>/YYYY/MM/DD/<name>.SAFE (None: not used)
l1c_cache = '/work/datalake/watcal/Sentinel-2/L1C'
restore_days = 3        # products on tier 3: days a restored copy stays available (10 at most)

# a Kerberos ticket cache readable from any node (KEYRING caches are per node, see the HPC wiki)
os.environ.setdefault('KRB5CCNAME', f'FILE:/home/tmp/krb5cc_{os.getuid()}')

### 1b. Datalake tools of `s3-env`

`run_in_s3_env` runs a short Python script with the Python of `s3-env`: the arguments are sent as JSON on its
standard input, and the last line it prints is read back as JSON. The credentials are passed to it as
`AWS_*` environment variables, never on the command line (visible to the other users of the node); the
`PYTHONPATH`, `PYTHONHOME`, `LD_LIBRARY_PATH` and `LD_PRELOAD` of this environment are removed, so that `s3-env`
runs with its own libraries. `role_credentials` gets the temporary credentials of a role with `assumerole`;
nothing secret is printed.

Common errors:

- `No Kerberos credentials available` (`SPNEGOExchangeError`): no valid ticket, run `kinit` (with
  `KRB5CCNAME` as in the setup);
- `OIDCException: Expected HTTP 302 response, got 200`: the identity provider of CNES did not accept the
  login, which comes before the role and its duration: the Kerberos ticket is missing or expired on this node
  (`klist`), held in a KEYRING cache (not seen by the compute nodes), or the request went to a proxy
  (`env | grep -i _proxy`). `connect` raises the error with these checks;
- `DurationSeconds` / `MaxSessionDuration`: the role refuses `role_hours`; the credentials are then
  fetched for 1 h (renewed more often). Any other error is raised as is;
- `AccessDenied`: your IPA account cannot take on this role (`roles_list.sh`);
- `s3-env not found`: set `S3_ENV` to the folder of the conda env (`conda env list` after `ml conda`).

In [ ]:
S3_ENV_CANDIDATES = ['/softs/rh8/conda-envs/awss3-env', '/work/softs/rh8/conda-envs/s3-env']
s3_env = os.environ.get('S3_ENV') or next((d for d in S3_ENV_CANDIDATES if os.path.isdir(d)),
                                          S3_ENV_CANDIDATES[0])      # conda env s3-env
S3_ENV_PYTHON = opj(s3_env, 'bin', 'python')


def run_in_s3_env(script, args=None, credentials=None, timeout=900):
    """Run a Python script in the conda env s3-env: `args` as JSON on its standard input, the last line printed
    read back as JSON. `credentials`: fsspec options whose key, secret and token are passed as AWS_* variables."""
    if not os.path.exists(S3_ENV_PYTHON):
        raise FileNotFoundError(f's3-env not found: {S3_ENV_PYTHON} (set S3_ENV)')
    env = {k: v for k, v in os.environ.items()
           if k not in ('PYTHONPATH', 'PYTHONHOME', 'LD_LIBRARY_PATH', 'LD_PRELOAD')}
    env['AWS_DEFAULT_REGION'] = 'us-east-1'
    if credentials and credentials.get('key'):
        env.update(AWS_ACCESS_KEY_ID=credentials['key'], AWS_SECRET_ACCESS_KEY=credentials['secret'],
                   AWS_SESSION_TOKEN=credentials['token'])
    proc = subprocess.run([S3_ENV_PYTHON, '-c', script], input=json.dumps(args or {}), env=env,
                          capture_output=True, text=True, timeout=timeout)
    lines = proc.stdout.strip().splitlines()
    if proc.returncode != 0 or not lines:
        error = '\n'.join(proc.stderr.strip().splitlines()[-3:])
        raise RuntimeError(f's3-env failed (exit {proc.returncode}): {error}')
    return json.loads(lines[-1])


CREDENTIALS_SCRIPT = """
import json, sys
from assumerole import assumerole
args = json.load(sys.stdin)
c = assumerole.getCredentials(args['arn'], Duration=args['duration'])
print(json.dumps({k: c[k] for k in ('AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_SESSION_TOKEN')}))
"""


def role_credentials(arn, hours=1):
    """Temporary credentials of a datalake role (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_SESSION_TOKEN),
    from assumerole in s3-env, valid `hours` hours (1 to 12)."""
    return run_in_s3_env(CREDENTIALS_SCRIPT, dict(arn=arn, duration=int(min(max(hours, 1), 12) * 3600)))

`datalake_storage_options` returns the options of the fsspec file system: endpoint, region and, with a
role, temporary credentials. `connect` (re)creates the file system `fs` used to list the images and gives
the new options to the batch: it is called again before each download, and renews the credentials when they
expire within `refresh_margin` minutes (every 45 min at most in `sync_l1c.py`).

In [ ]:
# the role refuses the duration asked (STS: DurationSeconds above the MaxSessionDuration of the role)
DURATION_REFUSED = re.compile(r'DurationSeconds|MaxSessionDuration', re.IGNORECASE)
# the login to the identity provider of CNES failed (Kerberos ticket missing, expired, or not sent)
LOGIN_FAILED = re.compile(r'OIDCException|SPNEGO|Kerberos|GSSError|krb5', re.IGNORECASE)


def credentials_error(e):
    """Error of role_credentials, with a hint when the login to the identity provider failed."""
    if not LOGIN_FAILED.search(str(e)):
        return e
    ccache = os.environ.get('KRB5CCNAME', '(not set)')
    return RuntimeError(
        f'{e}\n\nThe login of assumerole to the identity provider of CNES failed (an HTTP 200 instead of 302: the '
        'Kerberos ticket was not accepted). Check, on this node:\n'
        f'  - the ticket: `klist` with KRB5CCNAME={ccache}; if missing or expired, on a login node:\n'
        '      export KRB5CCNAME=FILE:/home/tmp/krb5cc_$(id -u); kinit; klist\n'
        '    (a KEYRING ticket cache is not seen by the compute nodes);\n'
        '  - a proxy (`env | grep -i _proxy`): the request to the identity provider may go to the proxy of CNES;\n'
        '  - from a login node: if it works there, fetch the credentials before submitting\n'
        '    (/softs/tools/datalake/use_role.sh -a <ARN> -t 43200, then role_arn = None).')


def datalake_storage_options():
    """fsspec options of the datalake: endpoint, region, and temporary credentials of `role_arn`."""
    global valid_hours
    options = dict(client_kwargs=dict(endpoint_url=endpoint_url, region_name='us-east-1'))
    if role_arn:
        try:
            credentials = role_credentials(role_arn, hours=role_hours)
            valid_hours = role_hours
        except RuntimeError as e:
            if role_hours <= 1 or not DURATION_REFUSED.search(str(e)):
                raise credentials_error(e) from None
            # duration longer than the maximum of the role: credentials of 1 h, renewed more often
            logging.warning(f'credentials for {role_hours} h refused ({e}): validity of 1 h')
            try:
                credentials = role_credentials(role_arn, hours=1)
            except RuntimeError as e1:
                raise credentials_error(e1) from None
            valid_hours = 1
        options.update(key=credentials['AWS_ACCESS_KEY_ID'], secret=credentials['AWS_SECRET_ACCESS_KEY'],
                       token=credentials['AWS_SESSION_TOKEN'])
    return options


connected_at, valid_hours = None, None


def connect(force=False):
    """(Re)connect to the datalake if the credentials expire within `refresh_margin` minutes."""
    global fs, storage_options, connected_at
    if not force and connected_at is not None and (
            not role_arn or time.monotonic() - connected_at < valid_hours * 3600 - refresh_margin * 60):
        return
    storage_options = datalake_storage_options()
    fs = fsspec.filesystem('s3', skip_instance_cache=True, **storage_options)
    connected_at = time.monotonic()
    if 'run' in globals():
        run.storage_options = storage_options
    logging.info(f'datalake: new credentials of {role_arn}, valid {valid_hours} h' if role_arn
                 else 'datalake: credentials of the environment')


connect(force=True)

### 1c. Look-up tables

GRS 2.2 reads its look-up tables from the `grsdata` folder of its `config.yml` (on TREX,
`/work/datalake/watcal/GRS/grsdata_v21`). Check they are found before a long batch:

In [ ]:
config_file = files('grs') / 'config.yml'
with open(config_file) as f:
    config = yaml.safe_load(f)['path']
print(f'configuration: {config_file}')
for key in ('toa_lut', 'trans_lut'):
    lut = opj(config['grsdata'], config[key])
    print(f"{key:9s} {lut}: {'found' if os.path.exists(lut) else 'MISSING, edit grsdata in config.yml'}")

## 2. Datalake: connection and layout

The buckets visible with the role (the Sentinel-2 ones are picked out), then the levels of `l1c_root`,
going down into the first tile (zone, band, square) and then into the latest folder, down to the products:
a check of `l1c_root` and `l1c_layout`. The storage class, when the listing gives it, shows the products on
tier 3.

In [ ]:
connect()
try:
    buckets = [os.path.basename(b.rstrip('/')) for b in fs.ls('', detail=False)]
    print(f'{len(buckets)} buckets visible; Sentinel-2:',
          [b for b in buckets if re.search(r'sentinel-?2|^s2', b, re.IGNORECASE)])
except Exception as e:
    print(f'buckets not listed ({type(e).__name__}: {e})')

path = l1c_root.removeprefix('s3://').rstrip('/')
for level in range(8):                     # e.g. <zone>/<band>/<square>/<year>/<month>/<day>/<product>
    entries = sorted(fs.ls(path, detail=True), key=lambda e: e['name'])
    names = [os.path.basename(e['name'].rstrip('/')) for e in entries]
    print(f'{path}/: {len(entries)} entries, e.g. {names[:4]}{" ... " + names[-1] if len(names) > 4 else ""}')
    products = [e for e, n in zip(entries, names) if n.startswith('S2')]
    if products:
        for e in products[:3]:
            print(f"  {e['name']}  {e['type']}  {e.get('size', 0) / 2**20:.0f} MB  "
                  f"storage class: {e.get('StorageClass', '-')}")
        break
    tile_dirs = [e for e, n in zip(entries, names)        # first tile: 28PFV, or 28 / P / FV
                 if n in (tiles[0], 'T' + tiles[0], tiles[0][:2], tiles[0][2], tiles[0][3:])]
    if not entries:
        break
    path = (tile_dirs or entries)[-1]['name'].rstrip('/')

## 3. List the L1C images

{func}`~grstbx.batch.list_images` needs the exact `<tile>/YYYY/MM/DD` tree and reads the cloud cover of
every image of the tile. On the datalake, with years of archive per tile, this is slow. Here:

- the products are found with `l1c_layout`, year by year, and parsed from their **names** (date, tile,
  baseline): the folder layout does not matter;
- a datatake found more than once (e.g. reprocessed with baseline `N0500` and `N0511`) keeps its
  **latest** version (highest baseline, then latest processing time);
- the **cloud cover** (`MTD_MSIL1C.xml`) is read only for the images of the date range, 16 at a time.
  For a `.zip`, only the metadata file is read from the archive, not the whole archive;
- an image already synchronised in `l1c_cache` (`local`) is read from there;
- the products on **tier 3** cannot be read before they are restored: their cloud cover is left `NaN` until
  then (section 4b). `tier3` is only what the listing tells (its storage class, if given); section 4b checks
  each product with `head_object`.

The table has the columns {class}`~grstbx.batch.Batch` expects: `tile`, `name` (always `<id>.SAFE`, so
that the products are named as with the `grs` command line), `path` (URL of the product in the bucket)
and `cloud_cover`.

In [ ]:
S2_NAME = re.compile(r'^(?P<id>S2[A-D]_MSIL1C_(?P<date>\d{8})T\d{6}_N(?P<baseline>\d{4})_R\d{3}_'
                     r'T(?P<tile>\d{2}[A-Z]{3})_(?P<ptime>\d{8}T\d{6}))(?P<ext>\.SAFE(\.zip)?|\.zip)$')
CLOUD = re.compile(rb'<Cloud_Coverage_Assessment>\s*([\d.]+)')


def l1c_cloud_cover(url):
    """Cloud cover (0-1) of an L1C product of the bucket, .SAFE folder or .zip archive; NaN if not found."""
    try:
        path = url.removeprefix('s3://')
        if path.endswith('.zip'):
            with fs.open(path, 'rb', block_size=2**20) as f, zipfile.ZipFile(f) as z:
                member = next(m for m in z.namelist() if m.endswith('MTD_MSIL1C.xml'))
                text = z.read(member)
        else:
            text = fs.cat_file(f'{path}/MTD_MSIL1C.xml')
        return float(CLOUD.search(text).group(1)) / 100
    except Exception as e:      # e.g. archived on tier 3: read after its restoration (section 4b)
        logging.debug(f'cloud cover of {url}: {type(e).__name__}: {e}')
        return np.nan


def cached_l1c(tile, date, name):
    """Image already synchronised in `l1c_cache` (<tile>/YYYY/MM/DD/<name>.SAFE), else None."""
    if not l1c_cache:
        return None
    path = opj(l1c_cache, tile, f'{pd.Timestamp(date):%Y/%m/%d}', name)
    return path if os.path.isfile(opj(path, 'MTD_MSIL1C.xml')) else None


def list_datalake_l1c(tiles, start, end):
    """L1C images of the datalake for the tiles and date range (inclusive), one per datatake."""
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    rows = []
    for tile in tiles:
        for year in range(start.year, end.year + 1):
            pattern = l1c_layout.format(root=l1c_root.rstrip('/'), tile=tile, zone=tile[:2], band=tile[2],
                                        square=tile[3:], year=year)
            for path, info in fs.glob(pattern.removeprefix('s3://'), detail=True).items():
                match = S2_NAME.match(os.path.basename(path.rstrip('/')))
                if match is None or match['tile'] != tile:
                    continue
                tier3 = 'glacier' in str(info.get('StorageClass', '')).lower()
                rows.append(dict(date=pd.Timestamp(match['date']), tile=tile, name=match['id'] + '.SAFE',
                                 path='s3://' + path, tier3=tier3, baseline=match['baseline'],
                                 ptime=match['ptime'], datatake=match['id'][:26]))   # S2x_MSIL1C_YYYYMMDDTHHMMSS
    columns = ['date', 'tile', 'name', 'path', 'tier3', 'baseline', 'ptime', 'datatake']
    df = pd.DataFrame(rows, columns=columns)
    df = df[(df.date >= start) & (df.date <= end)]
    n_found = len(df)
    df = (df.sort_values(['baseline', 'ptime'])
            .drop_duplicates(['tile', 'datatake'], keep='last'))     # latest version of each datatake
    print(f'{n_found} products found, {n_found - len(df)} older versions of a datatake dropped')
    df['local'] = [cached_l1c(t, d, n) for t, d, n in zip(df['tile'], df['date'], df['name'])]
    cached = df['local'].notna()
    df['cloud_cover'] = np.nan
    # each .loc assignment is guarded: some pandas versions refuse an empty value for an empty selection
    if cached.any():
        df.loc[cached, 'cloud_cover'] = df.loc[cached, 'local'].map(batch.cloud_cover).astype(float)
    readable = ~df['tier3'] & ~cached
    if readable.any():
        with ThreadPoolExecutor(16) as pool:
            df.loc[readable, 'cloud_cover'] = pd.Series(list(pool.map(l1c_cloud_cover, df.loc[readable, 'path'])),
                                                    index=df.index[readable], dtype=float)
    print(f'{cached.sum()} images already in {l1c_cache}; '
          f'{df.cloud_cover.isna().sum()} cloud covers not read yet (archived on tier 3?)')
    return df.drop(columns=['baseline', 'ptime', 'datatake']).set_index('date').sort_index()


t0 = time.perf_counter()
selection = list_datalake_l1c(tiles, start_date, end_date)
print(f'Found {len(selection)} L1C scene(s) in the datalake, listed in {time.perf_counter() - t0:.0f} s')
selection.groupby('tile').agg(images=('name', 'size'), cached=('local', 'count'), on_tier3=('tier3', 'sum'),
                              mean_cloud_cover=('cloud_cover', 'mean')).round(2)

In [ ]:
selection[['tile', 'name', 'local', 'tier3', 'cloud_cover']]

## 4. Plan: auxiliary data, product names and state

{meth}`Batch.plan_level1 <grstbx.batch.Batch.plan_level1>` adds the DEM, the L2A and L2B paths and what
is already done. The CAMS file of each image is then the first of `cams_file_templates` found for its day
(`cams_ok`: one was found).

- `l2a_done`: the NetCDF L2A product is complete. With `quick_check`, from file metadata only: `<name>.nc`
  larger than 1 MB and `<name>_anc.nc` present (as `process_l2a.py` of the guide); otherwise both files are
  opened and `Rrs` and `mask` looked for;
- `l2b_done`: the STAC Item of the L2B product exists (GRSl2bgen writes it last);
- `run_l2a`, `run_l2b`: the steps to run. An existing L2A product is reused for the L2B step.

`TrexBatch` is {class}`~grstbx.batch.Batch` with these metadata-only checks: opening thousands of NetCDF
files at the start of a batch floods the Lustre metadata servers (guide, troubleshooting matrix).

```
S2A_MSIL1C_20250602T105041_N0511_R051_T31TCJ_20250602T143513.SAFE              L1C (datalake)
S2A_MSIL2AGRS_20250602T105041_N0511_R051_T31TCJ_20250602T143513_V2.2.x.nc      L2A
S2A_MSIL2B_20250602T105041_N0511_R051_T31TCJ_20250602T143513_V2.2.x.nc         L2B
```

In [ ]:
def quick_l2a_complete(path):
    """NetCDF L2A product of GRS 2.2 complete, from file metadata only: <name>.nc larger than 1 MB and
    <name>_anc.nc present."""
    path = str(path)
    try:
        return os.path.getsize(path) > 2**20 and os.path.exists(path[:-len('.nc')] + '_anc.nc')
    except OSError:
        return False


class TrexBatch(batch.Batch):
    """Batch with the completeness of the L2A products checked from file metadata only (`quick_check`)."""

    def __init__(self, *args, quick_check=True, **kwargs):
        super().__init__(*args, **kwargs)
        self.quick_check = quick_check

    def _l2a_complete(self, path):
        return quick_l2a_complete(path) if self.quick_check else super()._l2a_complete(path)


def cams_file(date):
    """First CAMS file of `cams_file_templates` found for the day, else the first template (missing)."""
    names = [t.format(date=pd.Timestamp(date)) for t in cams_file_templates]
    return next((n for n in names if os.path.exists(n)), names[0])


run = TrexBatch(
    l2a_dir, l2b_dir, cams_file_template=cams_file_templates[0], dem_file_template=dem_file_template,
    # GRS 2.2: NetCDF L2A product + _anc.nc
    resolution=resolution, l2a_ext='.nc', max_cloud_cover=max_cloud_cover,
    opac_model=opac_model, scale_aot=scale_aot, levname=levname, suffix=suffix,
    # GRSl2bgen: NetCDF L2B product and its STAC Item
    l2b_ext='.nc', pyramid=False, stac=True, l2a_stac=False, persist_input=persist_input,
    keep_l2a=True, overwrite_l2a=overwrite_l2a, overwrite_l2b=overwrite_l2b, quick_check=quick_check,
    # L1C read from the datalake bucket, copied to work_dir image by image
    work_dir=work_dir, storage_options=storage_options,
    # profiles of the images: <batch_dir>/profiles/<tile>/<image>_profile.json
    profile_dir=opj(batch_dir, 'profiles'))

plan = run.plan_level1(selection)
plan['cams_file'] = [cams_file(date) for date in plan.index]
plan['cams_ok'] = plan['cams_file'].map(os.path.exists)
print(run.describe(plan))
missing_cams = sorted(set(plan.loc[plan.run_l2a & ~plan.cams_ok, 'cams_file']))
if missing_cams:
    print(f'{len(missing_cams)} days without CAMS file, e.g.\n  ' + '\n  '.join(missing_cams[:5]))
print('tiles without DEM:', sorted(set(plan.loc[plan.dem_file.isna(), 'tile'])) or 'none')
plan[['tile', 'name', 'tier3', 'cloud_cover', 'cams_ok', 'no_water', 'l2a_done', 'l2b_done', 'run_l2a', 'run_l2b']]

### 4b. Products on tier 3: restoration

Every product to download (L2A to compute, not in `l1c_cache`) is checked with `head_object`, as the
listing may not give the storage class:

| `restore` | meaning | what happens |
|---|---|---|
| `available` | not (or no longer) on tier 3 | processed |
| `restored` | restored copy available (`ongoing-request="false"`) | processed |
| `restoring` | restoration in progress (`ongoing-request="true"`) | skipped, status `tier 3` |
| `archived` | no restoration requested | restoration **requested** if `request_restore`, skipped |

These checks and the restoration requests run in `s3-env` (one call for all the products), with `boto3`
and, on the OT account, the `amaltheerestore` library (outside the proxy, as `sync_l1c.py` does); on other
accounts, with `restore_object` (if the role has that permission). A restoration takes hours: run the batch
again later (or submit the SLURM array again) to process the restored images. Their cloud cover is read here,
once restored, so a cloudy image is only known to be cloudy after its restoration.

In [ ]:
request_restore = True

TIER3_SCRIPT = """
import json, os, sys
import boto3
args = json.load(sys.stdin)
s3 = boto3.client('s3', endpoint_url=args['endpoint'], region_name='us-east-1')
states = {}
for url in args['urls']:
    bucket, key = url.split('://', 1)[-1].split('/', 1)
    try:
        head = s3.head_object(Bucket=bucket, Key=key)
        if 'glacier' not in str(head.get('StorageClass', '')).lower():
            state = 'available'
        elif head.get('Restore') is None:
            state = 'archived'
        else:
            state = 'restoring' if 'ongoing-request="true"' in head['Restore'] else 'restored'
        if state == 'archived' and args['request']:
            if args['ot']:
                from amaltheerestore import amaltheerestore
                # the restoration service is called outside the proxy (as sync_l1c.py of the TREX guide)
                proxies = {k: os.environ.pop(k, None) for k in ('http_proxy', 'https_proxy', 'HTTP_PROXY', 'HTTPS_PROXY')}
                try:
                    amaltheerestore.amalthee_restore_object(bucket, key, args['days'], 0)
                finally:
                    os.environ.update({k: v for k, v in proxies.items() if v})
            else:
                s3.restore_object(Bucket=bucket, Key=key, RestoreRequest={'Days': args['days']})
            state = 'requested'
    except Exception as e:
        state = 'error: %s: %s' % (type(e).__name__, e)
    states[url] = state
print(json.dumps(states))
"""


def tier3_states(urls):
    """State of objects possibly on tier 3 ('available', 'restored', 'restoring', 'archived', 'requested',
    'error: ...'), restorations requested if `request_restore`, in s3-env."""
    connect()
    args = dict(urls=list(urls), endpoint=endpoint_url, request=request_restore, days=restore_days,
                ot=bool(role_arn and OT_ACCOUNT in role_arn))
    return run_in_s3_env(TIER3_SCRIPT, args, credentials=storage_options)


plan['restore'] = np.where(plan['tier3'], 'unknown', 'available')
check = plan['run_l2a'] & plan['cams_ok'] & plan['local'].isna()
if check.any():
    states = tier3_states(plan.loc[check, 'path'])
    for url, state in states.items():
        if state.startswith('error'):
            logging.warning(f'{url}: {state}')
    plan.loc[check, 'restore'] = pd.Series([states[url] for url in plan.loc[check, 'path']],
                                           index=plan.index[check], dtype=object)
    # cloud cover of the restored products, now readable
    restored = check & plan['restore'].isin(['restored', 'available']) & plan['cloud_cover'].isna()
    if restored.any():
        with ThreadPoolExecutor(16) as pool:
            plan.loc[restored, 'cloud_cover'] = pd.Series(list(pool.map(l1c_cloud_cover, plan.loc[restored, 'path'])),
                                                          index=plan.index[restored], dtype=float)
    plan['cloudy'] = plan['cloud_cover'] >= max_cloud_cover
plan['available'] = plan['restore'].isin(['available', 'restored'])
print('products to compute, by state on the datalake:',
      plan.loc[plan.run_l2a, 'restore'].value_counts().to_dict())

## 5. Run the batch

For each image with a step to run, {func}`process_image`:

1. copies the L1C product from the bucket to `work_dir`. A `.SAFE` folder is copied as is by
   {meth}`Batch.process <grstbx.batch.Batch.process>`; a `.zip` archive is downloaded and extracted
   first, since GRS reads `.SAFE` folders. An image synchronised in `l1c_cache` is read from there;
2. runs GRS and then GRSl2bgen with {meth}`Batch.process <grstbx.batch.Batch.process>`. The
   products go straight to `/work/datalake`. A failing step is logged, its partial output removed (the
   image is retried at the next run), and the batch goes on;
3. removes the local copy, and writes the profile of the image (time and memory of each step).

**Several images at a time.** `workers` images are processed at the same time. Each one has a thread that
downloads its L1C product and then waits for the child process of its steps (`Batch.process` runs them in
a process of their own, `isolate`): the computation is in the child processes, whose memory is given back
after each image, and a child killed by the system (out of memory) only fails its own image. The
credentials of the datalake are renewed before each image is started. Raise `workers` with the memory:
the profiles (section 6) give the peak memory of each step. The images may end in any order; the summary
keeps the order of the plan.

The images whose steps cannot run (product done, too cloudy, no CAMS file, on tier 3 and not yet restored)
are not downloaded. Once `max_hours` has elapsed, no new image is started: the remaining ones are
`not started`, and are processed by the next run.

**Images without clear water.** When every water pixel of an image is cloudy or outside the swath, GRS 2.2
stops with `ValueError: zero-size array to reduction operation fmin which has no identity` (guide,
section 4.4.1). Such an image is not an error of the batch: `process_image` writes the marker
`<name>_no_water.txt` in place of the product (as GRS 3.x does for an image without water), so it is reported
`no_water` and not tried again. Remove the marker to try it again.

The lines printed for each image follow `process_l2a.py` of the guide (`Processing`, `Finished`, `Skipped`,
`Error processing`), so that `status.sh` of the guide can follow a SLURM job.

In [ ]:
FMIN_ERROR = ('fmin', 'no identity')     # GRS stops on an image without clear water pixel


def stage_zip(url, name):
    """Download and extract an L1C .zip archive of the bucket; return the local .SAFE folder."""
    local_dir = opj(run.work_dir, 'L1', name.removesuffix('.SAFE'))
    shutil.rmtree(local_dir, ignore_errors=True)
    os.makedirs(local_dir)
    archive = opj(local_dir, os.path.basename(url))
    fs.get(url.removeprefix('s3://'), archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(local_dir)
    os.remove(archive)
    safe = [opj(local_dir, d) for d in os.listdir(local_dir) if d.endswith('.SAFE')]
    if len(safe) != 1:
        raise FileNotFoundError(f'no .SAFE folder in {url}')
    return safe[0], local_dir


def mark_no_clear_water(row, record):
    """An L2A step that failed on an image without clear water pixel: marker <name>_no_water.txt in place of
    the product, so that the image is skipped by the next runs (status 'no_water')."""
    if record['status'] != 'L2A failed' or not all(s in record['error'] for s in FMIN_ERROR):
        return record
    marker = batch.no_water_marker(row['l2a'])
    os.makedirs(os.path.dirname(marker), exist_ok=True)
    with open(marker, 'w') as f:
        f.write(f"no clear water pixel in {row['name']} (GRS {grs.__version__}: {record['error']}); "
                'remove this file to process the image again\n')
    return record | dict(status='no_water', l2a='no_water')


def process_image(row):
    """Steps of one image of the plan, with the L1C product staged from the datalake."""
    blocked = row['run_l2a'] and (not row['cams_ok'] or row['cloudy'])
    if not row['run_l2a'] or blocked:
        return run.process(row)           # skipped, blocked, or L2B from an existing L2A: no download
    if row['local']:                      # already synchronised: processed from l1c_cache
        row = row.copy()
        row['path'] = row['local']
        return mark_no_clear_water(row, run.process(row))
    if not row['available']:              # on tier 3, not restored yet: processed by a next run
        return dict(tile=row['tile'], name=row['name'], status='tier 3', l2a='', l2b='',
                    error=f"restoration {row['restore']}", duration_s=0.)
    staged = None
    try:
        if row['path'].endswith('.zip'):
            row = row.copy()
            try:
                row['path'], staged = stage_zip(row['path'], row['name'])
            except Exception as e:
                return dict(tile=row['tile'], name=row['name'], status='L1C failed', l2a='', l2b='',
                            error=f'{type(e).__name__}: {e}', duration_s=0.)
        return mark_no_clear_water(row, run.process(row))   # downloads a .SAFE URL itself, removes its copy
    finally:
        if staged:
            shutil.rmtree(staged, ignore_errors=True)


def report(i, n, row, record, elapsed):
    """Line of an image, worded as process_l2a.py of the guide (read by status.sh)."""
    name, status = row['name'].removesuffix('.SAFE'), record['status']
    head = f'  [{i}/{n}] [{pd.Timestamp.now():%H:%M:%S}]'
    if status == 'done':
        print(f"{head} Finished {name} in {record['duration_s']:.1f}s (total {elapsed:.2f} h)")
    elif status in ('no_water', 'cloudy', 'no_cams', 'tier 3'):
        print(f"{head} Skipped {name}: {status} {record['error']}".rstrip())
    elif status != 'skipped':
        print(f"{head} Error processing {name}: {status}: {record['error'][:300]}")


def failed(row, e):
    """Record of an image whose processing raised an exception (not caught by Batch.process)."""
    return dict(tile=row['tile'], name=row['name'], status='L2A failed', l2a='', l2b='',
                error=f'{type(e).__name__}: {e}', duration_s=0.)


def run_batch(plan, max_hours=None, workers=1):
    """Images of the plan, `workers` at a time: each in a thread that stages its L1C product and waits for the
    child process of its steps (Batch.process with isolate). Credentials renewed before each image is started;
    no image started past `max_hours`. Summary in the order of the plan."""
    from concurrent.futures import FIRST_COMPLETED, wait

    workers = max(1, int(workers))
    if workers > 1 and not run.isolate:
        raise ValueError('workers > 1 needs run.isolate (each image in its own process)')
    t0 = time.perf_counter()
    rows = list(plan.iterrows())
    records = [None] * len(rows)
    todo = plan[plan.run_l2a | plan.run_l2b]
    print(f'  Found {len(todo)} L1C scene(s) to process (out of {len(plan)}), {workers} at a time.')

    def hours():
        return (time.perf_counter() - t0) / 3600

    def finish(i, future=None, record=None):
        date, row = rows[i]
        if future is not None:
            try:
                record = future.result()
            except Exception as e:
                record = failed(row, e)
        records[i] = dict(date=date) | record
        report(i + 1, len(rows), row, record, hours())

    def collect(running, return_when):
        done, _ = wait(running, return_when=return_when)
        for future in done:
            finish(running.pop(future), future)

    with ThreadPoolExecutor(workers) as pool:
        running = {}                      # future -> index of the image in the plan
        for i, (date, row) in enumerate(rows):
            if not (row['run_l2a'] or row['run_l2b']):
                finish(i, record=process_image(row))      # nothing to run: no download, no process
                continue
            while len(running) >= workers:                # a free slot
                collect(running, FIRST_COMPLETED)
            if max_hours is not None and hours() > max_hours:
                finish(i, record=dict(tile=row['tile'], name=row['name'], status='not started', l2a='', l2b='',
                                      error=f'time budget of {max_hours} h reached', duration_s=0.))
                continue
            print(f"  [{i + 1}/{len(plan)}] [{pd.Timestamp.now():%H:%M:%S}] Processing {row['name'].removesuffix('.SAFE')}...",
                  flush=True)
            if row['run_l2a'] and row['cams_ok'] and not row['cloudy']:
                connect()                 # renew the temporary credentials before downloading
            running[pool.submit(process_image, row)] = i
        while running:
            collect(running, FIRST_COMPLETED)
    columns = ['date', 'tile', 'name', 'status', 'l2a', 'l2b', 'error', 'duration_s']
    return pd.DataFrame(records, columns=columns).set_index('date')


os.makedirs(run.work_dir, exist_ok=True)
summary = run_batch(plan, max_hours=max_hours, workers=workers)
summary.groupby(['tile', 'status']).size().unstack(fill_value=0)


## 6. Summary and profiles

The summary is saved in `batch_dir`, one CSV per tile, out of the product folders:
`batch/summaries/<tile>/l1c_l2b_<tile>_<start>-<end>_<run time>.csv`, with `_job<id>_<task>` for a task
of the SLURM array (as its log, `logs/l2a_l2b_<job>_<task>.out`). The profiles of the images are in
`batch/profiles/<tile>/`. Failed images (see the `log_file.log` of the L2A folder and the
`<name>.log` of the L2B folder) are retried by running the batch again.

In [ ]:
def save_summaries(summary):
    """Summary of the run, one CSV per tile, out of the product folders:
    <batch_dir>/summaries/<tile>/l1c_l2b_<tile>_<start>-<end>_<run time>[_job<id>_<task>].csv
    (the SLURM job and task in the name, as in the log logs/l2a_l2b_<job>_<task>.out)."""
    stamp = f'{pd.Timestamp.now():%Y%m%dT%H%M%S}'
    job = os.environ.get('SLURM_ARRAY_JOB_ID') or os.environ.get('SLURM_JOB_ID')
    if os.environ.get('GRS_BATCH_JOB') and job:
        task = os.environ.get('SLURM_ARRAY_TASK_ID')
        stamp += f'_job{job}' + (f'_{task}' if task else '')
    period = f'{pd.Timestamp(start_date):%Y%m%d}-{pd.Timestamp(end_date):%Y%m%d}'
    paths = []
    for tile, rows in summary.groupby('tile'):
        folder = opj(batch_dir, 'summaries', tile)
        os.makedirs(folder, exist_ok=True)
        paths.append(opj(folder, f'l1c_l2b_{tile}_{period}_{stamp}.csv'))
        rows.to_csv(paths[-1])
    return paths


for path in save_summaries(summary):
    print(f'summary saved to {path}')
print(f"{(summary.status == 'tier 3').sum()} images waiting for their restoration from tier 3: run again later")
summary[summary.status.str.contains('failed|no_cams|not started')][['tile', 'status', 'error', 'name']]

In [ ]:
profiles = batch.read_profiles([p for p in plan['profile'] if p and os.path.exists(p)])
done = profiles[profiles.status == 'done'] if len(profiles) else profiles
if len(done):
    print(done.groupby('step').agg(images=('image', 'size'), wall_s=('wall_s', 'mean'),
                                   cpu_load=('cpu_load', 'mean'), peak_mb=('rss_peak_mb', 'max')).round(1))

## 7. Quick look at the products of one image

The image with the most valid water pixels among those with both products: true-colour $R_{rs}$ of the
L2A product (`mask == 0`) and three parameters of the L2B product. Skipped in the batch tasks of the SLURM array (section 8).

In [ ]:
def open_product(path):
    """L2A or L2B product, NetCDF file or Zarr store (full-resolution level of a pyramid)."""
    if str(path).endswith('.zarr'):
        groups = [d for d in os.listdir(path) if d.isdigit()]
        return xr.open_zarr(path, group='0' if groups else None, decode_coords='all')
    return xr.open_dataset(path, decode_coords='all')


both = plan[plan['l2a'].map(run._l2a_complete) & plan['l2b'].map(os.path.exists)]
if os.environ.get('GRS_BATCH_JOB') or both.empty:
    print('no quick look' + ('' if os.environ.get('GRS_BATCH_JOB') else ': no image with both products'))
else:
    with ThreadPoolExecutor(8) as pool:
        valid = list(pool.map(lambda p: float((open_product(p).mask[::10, ::10] == 0).mean()), both['l2a']))
    row = both.iloc[int(np.argmax(valid))]
    l2a, l2b = open_product(row['l2a']), open_product(row['l2b'])
    params = [('Chla_OC2nasa', dict(cmap='Spectral_r', vmin=0, vmax=10)),
              ('TURB_dogliotti', dict(cmap='Spectral_r', vmin=0, vmax=20)),
              ('owt_index_Ta2025', dict(cmap='tab20'))]
    params = [(p, kw) for p, kw in params if p in l2b][:3]
    fig, axs = plt.subplots(1, 1 + len(params), figsize=(6 * (1 + len(params)), 5.5))
    axs = np.atleast_1d(axs)
    l2a.Rrs.sel(wl=[665, 560, 490], method='nearest').where(l2a.mask == 0).plot.imshow(
        rgb='wl', robust=True, ax=axs[0])
    axs[0].set_title('L2A: Rrs, valid water pixels')
    for ax, (param, kwargs) in zip(axs[1:], params):
        l2b[param].squeeze().plot.imshow(ax=ax, cbar_kwargs={'shrink': 0.7}, **kwargs)
        ax.set_title(f"L2B: {param} ({l2b[param].attrs.get('units', '-')})")
    for ax in axs:
        ax.set_aspect('equal')
        ax.set_axis_off()
    fig.suptitle(row['name'])
    plt.tight_layout()

## 8. Large batches: SLURM job array on TREX

A notebook run interactively is limited by the session. For years of archive, run this notebook as a
**SLURM job array** on a **tile × month grid**, as `run_l2a_array_2d.slurm` of the guide: one task per tile and
month, the arguments being tiles, years and, optionally, months (`31TGM 31TFJ 2025 2026`, `31TGM 2025 6 7 8`
or `31TGM 2025 jun jul`). Run without `sbatch`, the script prints the size of the grid and the `sbatch`
command (dry run). The code cells of the notebook are run in order by `python` (no `nbformat` nor IPython needed;
the task stops at the first error, with exit code 1), so that its output goes live to the log of the task, `logs/l2a_l2b_<job>_<task>.out`, which `status.sh` of the guide can follow (`./status.sh l2a`).

Each task:

- 16 cores, 64 GB of memory and 4 h per task, as `run_l2a_array_2d.slurm` of the guide (a full tile at 20 m
  takes about 30 GB with GRS 2.2);
- with `workers` images at a time (section 0): 8 cores per image (16 at least) and
  `workers × mem_per_image` GB of memory (64 at least);
- sets `GRS_BATCH_JOB=1`, so that the notebook knows it runs as a batch task: no quick look, script not
  rewritten (`SLURM_JOB_ID` cannot tell, as JupyterHub on TREX also runs the Jupyter server in a SLURM job);
- uses the node-local `$TMPDIR` as `work_dir`;
- stops starting new images 30 min before its time limit (`GRS_MAX_HOURS`), so that no product is cut
  in the middle;
- skips what is already done: submit the same grid again to continue (e.g. after tier-3 restorations);
- follows the rules of the guide: `--export=none`, `unset PYTHONPATH`, `PYTHONNOUSERSITE=1`,
  `MALLOC_ARENA_MAX=2`, 8 threads for OpenMP / BLAS / GDAL, `GDAL_DISABLE_READDIR_ON_OPEN=EMPTY_DIR`, one log
  file per task (no concurrent writes to a shared file on Lustre), months forced to base 10 (`08`, `09`);
- renews the temporary credentials of the datalake role before they expire (section 1b),
  which needs a Kerberos ticket readable from the compute node. Before submitting, on a login node:
  `export KRB5CCNAME=FILE:/home/tmp/krb5cc_$(id -u); kinit; klist`. The ticket is valid for about a week.

The job runs in the conda env `conda_env` (`GRSv2` by default, `module load conda`); the notebook calls `s3-env` itself for the
credentials and the tier-3 restorations.

The cell below writes the script next to this notebook (not run in a SLURM job). Set the project account
(`-A`).

In [ ]:
notebook = os.path.abspath('grs_datalake_trex_batch.ipynb')    # this notebook
log_dir = os.path.join(os.path.dirname(notebook), 'logs')   # absolute: SLURM opens the log files before the script runs
script = f"""#!/bin/bash
#SBATCH --job-name=GRS2_L1C_L2B
#SBATCH --output={log_dir}/l2a_l2b_%A_%a.out
#SBATCH --error={log_dir}/l2a_l2b_%A_%a.err
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c {max(16, 8 * workers)}
#SBATCH --mem={max(64, workers * mem_per_image)}G
#SBATCH --time={hours:02d}:00:00
#SBATCH -A {account}
#SBATCH -p cpu2022
#SBATCH --export=none
# L1C (CNES datalake) -> L2A (GRS {grs.__version__}) -> L2B (GRSl2bgen), one task per tile and month.
# Usage: run_grs2_datalake_2d.slurm <tile> [tile...] <year> [year...] [month...]   (dry run: grid and sbatch command)
#        sbatch --array=0-<N-1>%8 run_grs2_datalake_2d.slurm <same arguments>

set -eo pipefail
export LC_ALL=C
cd "{os.path.dirname(notebook)}"    # folder of the notebook, wherever sbatch is run from
mkdir -p "{log_dir}"

TILES=(); YEARS=(); MONTHS=()
for arg in "$@"; do
    if [[ "$arg" =~ ^[0-9]{{4}}$ ]]; then
        YEARS+=("$arg")
    elif [[ "$arg" =~ ^[0-9]{{1,2}}$ ]]; then
        MONTHS+=( $(printf "%02d" "$(( 10#$arg ))") )      # base 10: 08 and 09 are not octal
    elif [[ "$arg" =~ ^[A-Za-z]{{3,9}}$ ]]; then
        m=$(date -d "1 $arg 2000" +%m 2>/dev/null || true)
        [[ -n "$m" ]] && MONTHS+=("$m")
    elif [[ "$arg" =~ ^[Tt]?[0-9]{{2}}[A-Za-z]{{3}}$ ]]; then
        RAW_TILE="${{arg#[Tt]}}"
        TILES+=("${{RAW_TILE^^}}")
    fi
done
[[ ${{#TILES[@]}} -eq 0 ]] && {{ echo "[-] Error: give tile(s), e.g. 31TGM"; exit 1; }}
[[ ${{#YEARS[@]}} -eq 0 ]] && {{ echo "[-] Error: give year(s), e.g. 2025"; exit 1; }}
[[ ${{#MONTHS[@]}} -eq 0 ]] && MONTHS=(01 02 03 04 05 06 07 08 09 10 11 12)
readarray -t YEARS < <(printf "%s\\n" "${{YEARS[@]}}" | sort -nu)
readarray -t MONTHS < <(printf "%s\\n" "${{MONTHS[@]}}" | sort -nu)
readarray -t TILES < <(printf "%s\\n" "${{TILES[@]}}" | sort -u)
PERIODS=()
for y in "${{YEARS[@]}}"; do for m in "${{MONTHS[@]}}"; do PERIODS+=("$y $m"); done; done
NUM_PERIODS=${{#PERIODS[@]}}
TOTAL_TASKS=$(( ${{#TILES[@]}} * NUM_PERIODS ))

if [[ -z "${{SLURM_ARRAY_TASK_ID:-}}" ]]; then        # dry run
    echo "  L1C-L2B grid: ${{#TILES[@]}} tile(s) x $NUM_PERIODS month(s) = $TOTAL_TASKS tasks"
    echo "Submit command:"
    echo "  sbatch --array=0-$((TOTAL_TASKS - 1))%8 $0 $*"
    exit 0
fi
[[ $SLURM_ARRAY_TASK_ID -ge $TOTAL_TASKS ]] && exit 0

CURRENT_TILE="${{TILES[$(( SLURM_ARRAY_TASK_ID / NUM_PERIODS ))]}}"
read -r CURRENT_YEAR CURRENT_MONTH <<< "${{PERIODS[$(( SLURM_ARRAY_TASK_ID % NUM_PERIODS ))]}}"
START_DATE="${{CURRENT_YEAR}}-${{CURRENT_MONTH}}-01"
END_DATE=$(date -d "$START_DATE + 1 month - 1 day" +%Y-%m-%d)
echo "[$(date +'%Y-%m-%d %H:%M:%S')] Job $SLURM_ARRAY_JOB_ID task $SLURM_ARRAY_TASK_ID: Tile $CURRENT_TILE ($START_DATE to $END_DATE) on $(hostname)"

source ~/.bashrc 2>/dev/null || true
module load conda
conda activate {conda_env}    # processing; the datalake steps call s3-env itself
unset PYTHONPATH
export PYTHONNOUSERSITE=1

export GRS_TILES=$CURRENT_TILE GRS_START=$START_DATE GRS_END=$END_DATE
export GRS_OUT_ROOT=${{GRS_OUT_ROOT:-{out_root}}}
export GRS_MAX_HOURS={hours - 0.5}       # no new image in the last 30 min
export GRS_WORKERS={workers}            # images at a time, 8 threads each
export DATALAKE_ROLE_ARN={role_arn or '<arn:aws:iam::account:role/role, from roles_list.sh>'}
export KRB5CCNAME=FILE:/home/tmp/krb5cc_$(id -u)   # Kerberos ticket for the credentials, readable from the node
export OMP_NUM_THREADS=8 OPENBLAS_NUM_THREADS=8 MKL_NUM_THREADS=8 GDAL_NUM_THREADS=8
export GDAL_CACHEMAX=4096 GDAL_DISABLE_READDIR_ON_OPEN=EMPTY_DIR
export MALLOC_ARENA_MAX=2
export MPLBACKEND=Agg
export GRS_BATCH_JOB=1       # batch task (JupyterHub on TREX also runs in a SLURM job)

# the code cells of the notebook, run in order by python (no nbformat nor IPython needed): exit code 1 at the
# first error, with its traceback in the .err log
if python -u - "{notebook}" <<'PYEOF'
import json, sys
path = sys.argv[1]
cells = [c for c in json.load(open(path))['cells'] if c['cell_type'] == 'code']
namespace = {{'__name__': '__main__'}}
for i, cell in enumerate(cells):
    exec(compile(''.join(cell['source']), f'{{path}} [code cell {{i + 1}}]', 'exec'), namespace)
PYEOF
then
    echo "[$(date +'%Y-%m-%d %H:%M:%S')] COMPLETED"
else
    echo "[$(date +'%Y-%m-%d %H:%M:%S')] FAILED"; exit 1
fi
"""
if os.environ.get('GRS_BATCH_JOB'):
    print('in a batch task: script not written')
else:
    script_file = opj(os.path.dirname(notebook), 'run_grs2_datalake_2d.slurm')
    os.makedirs(log_dir, exist_ok=True)    # must exist before sbatch
    with open(script_file, 'w') as f:
        f.write(script)
    os.chmod(script_file, 0o755)
    print(f'written {script_file}; dry run (grid and sbatch command):')
    print(subprocess.run([script_file, *tiles, *sorted({str(y) for y in range(pd.Timestamp(start_date).year,
                                                                         pd.Timestamp(end_date).year + 1)})],
                         capture_output=True, text=True).stdout)

For example, from the folder of the notebook on a TREX login node:

```bash
./run_grs2_datalake_2d.slurm 31TGM 31TFJ 2025            # dry run: 2 tiles x 12 months = 24 tasks
sbatch --array=0-23%8 run_grs2_datalake_2d.slurm 31TGM 31TFJ 2025
./status.sh l2a                                   # progress of each task (status.sh of the guide)
tail -f logs/l2a_l2b_<job>_*.out                  # live log of the tasks
```

A task left `not started` images (time budget) or images waiting for their restoration from tier 3: submit
the same grid again, the products done are skipped. The tasks of a year run as independent jobs: limit the
concurrent ones (`%8`) to share the datalake bandwidth and the Lustre storage.